In [10]:
import os
import operator
from functools import reduce
from typing import List, Literal

from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

In [11]:
# ---------------------------------------------------------
# Input Schemas for Strict Validation
# ---------------------------------------------------------
class MathInput(BaseModel):
    numbers: List[float] = Field(description="A list of numbers to operate on.")

class PowerInput(BaseModel):
    base: float = Field(description="The base number.")
    exponent: float = Field(description="The exponent to raise the base to.")

class WikiInput(BaseModel):
    query: str = Field(description="The search query for Wikipedia.")

In [12]:
# ---------------------------------------------------------
# Tool Definitions
# ---------------------------------------------------------
@tool(args_schema=MathInput)
def add_numbers(numbers: List[float]) -> float:
    """Adds a list of numbers together."""
    return sum(numbers)

@tool(args_schema=MathInput)
def subtract_numbers(numbers: List[float]) -> float:
    """Subtracts numbers sequentially: first - second - third..."""
    if not numbers: return 0.0
    return reduce(operator.sub, numbers)

@tool(args_schema=MathInput)
def multiply_numbers(numbers: List[float]) -> float:
    """Calculates the product of a list of numbers."""
    if not numbers: return 1.0
    return reduce(operator.mul, numbers)

@tool(args_schema=MathInput)
def divide_numbers(numbers: List[float]) -> float:
    """Divides the first number by subsequent numbers."""
    if not numbers: return 0.0
    result = numbers[0]
    for num in numbers[1:]:
        if num == 0:
            # Raising an error lets the ToolNode catch it and pass the failure back to the LLM
            raise ValueError("Division by zero is mathematically undefined.")
        result /= num
    return result

@tool(args_schema=PowerInput)
def calculate_power(base: float, exponent: float) -> float:
    """Calculates base raised to the power of exponent."""
    return base ** exponent

@tool(args_schema=WikiInput)
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual data (like populations or distances)."""
    # Optimized to prevent context window bloat
    wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
    return wiki.run(query)

In [13]:
# Initialize LLM
# Ensure OPENAI_API_KEY is in your environment variables
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Bind tools to the LLM
tools = [
    add_numbers, 
    subtract_numbers, 
    multiply_numbers, 
    divide_numbers, 
    calculate_power, 
    search_wikipedia
]
llm_with_tools = llm.bind_tools(tools)

# Define System Prompt
sys_msg = SystemMessage(
    content="You are a precise mathematical assistant. Always use your tools for calculations and retrieving facts. Do not guess."
)

In [14]:
# ---------------------------------------------------------
# Node Functions
# ---------------------------------------------------------
def agent_node(state: MessagesState):
    """Invokes the LLM to decide the next action."""
    messages = state['messages']
    
    # Inject system message if it isn't already the first message
    if not messages or not isinstance(messages[0], SystemMessage):
        messages = [sys_msg] + messages
        
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def route_logic(state: MessagesState) -> Literal["tools", END]:
    """Determines whether to execute a tool or end the workflow."""
    last_message = state['messages'][-1]
    
    # If the LLM generated tool calls, route to the tools node
    if last_message.tool_calls:
        return "tools"
    
    # Otherwise, return to the user
    return END

In [15]:
# ---------------------------------------------------------
# Graph Compilation
# ---------------------------------------------------------
workflow = StateGraph(MessagesState)

# Add Nodes
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(tools)) # ToolNode automatically handles execution and error catching

# Add Edges
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", route_logic)
workflow.add_edge("tools", "agent")

# Compile into a runnable application
math_pipeline = workflow.compile()
print("Systems-first modular LangGraph pipeline initialized successfully.")

Systems-first modular LangGraph pipeline initialized successfully.


In [16]:
# Test the pipeline across various domains
queries = [
    "What is 150 minus 50, then multiplied by 2?",
    "What is the current population of France? Divide that number by 2.",
    "What is 5 to the power of 3?"
]

for q in queries:
    print(f"\n[User Query]: {q}")
    
    # Invoke the graph
    response = math_pipeline.invoke({"messages": [HumanMessage(content=q)]})
    
    # The final message content is the agent's synthesized answer
    print(f"[Agent Response]: {response['messages'][-1].content}")
    print("-" * 50)


[User Query]: What is 150 minus 50, then multiplied by 2?
[Agent Response]: 150 minus 50 is 100, and then multiplying that result by 2 gives 200.
--------------------------------------------------

[User Query]: What is the current population of France? Divide that number by 2.
[Agent Response]: The current population of France is approximately 69,081,996. Dividing that number by 2 gives 34,540,998.
--------------------------------------------------

[User Query]: What is 5 to the power of 3?
[Agent Response]: 5 to the power of 3 is 125.
--------------------------------------------------
